## 1. Préparation du fichier

Il faut récupérer la séquence de 28AA de la boucle variable, et l'ajouter au reste de la séquence de référence.

Ensuite, il faut préparer le fichier d'apprentissage en ajoutant toutes les séquences dans une liste immense, et tous les booléens dans une autre liste.

Donc entrée : Le ficher CSV
Sortie : Deux listes


In [7]:


VP1_WT_1 = "MAADGYLPDWLEDTLSEGIRQWWKLKPGPPPPKPAERHKDDSRGLVLPGYKYLGPFNGLDKGEPVNEADAAALEHDKAYDRQLDSGDNPYLKYNHADAEFQERLKEDTSFGGNLGRAVFQAKKRVLEPLGLVEEPVKTAPGKKRPVEHSPVEPDSSSGTGKAGQQPARKRLNFGQTGDADSVPDPQPLGQPPAAPSGLGTNTMATGSGAPMADNNEGADGVGNSSGNWHCDSTWMGDRVITTSTRTWALPTYNNHLYKQISSQSGASNDNHYFGYSTPWGYFDFNRFHCHFSPRDWQRLINNNWGFRPKRLNFKLFNIQVKEVTQNDGTTTIANNLTSTVQVFTDSEYQLPYVLGSAHQGCLPPFPADVFMVPQYGYLTLNNGSQAVGRSSFYCLEYFPSQMLRTGNNFTFSYTFEDVPFHSSYAHSQSLDRLMNPLIDQYLYYLSRTNTPSGTTTQSRLQFSQAGASDIRDQSRNWLPGPCYRQQRVSKTSADNNNSEYSWTGATKYHLNGRDSLVNPGPAMASHKDDEEKFFPQSGVLIFGKQGSEKTNVDIEKVMIT"

WT_boucle28 = "DEEEIRTTNPVATEQYGSVSTNLQRGNR"

VP1_WT_2 = "QAATADVNTQGVLPGMVWQDRDVYLQGPIWAKIPHTDGHFHPSPLMGGFGLKHPPPQILIKNTPVPANPSTTFSAAKFASFITQYSTGQVSVEIEWELQKENSKRWNPEIQYTSNYNKSVNVDFTVDTNGVYSEPRPIGTRYLTRNL"

VP1_WT = VP1_WT_1 + WT_boucle28 + VP1_WT_2

import pandas as pd
import numpy as np
data = pd.read_csv('Variants.csv')

min_valide = data[data['viral_selection'] != -np.inf]['viral_selection'].min()
max_valide = data[data['viral_selection'] != -np.inf]['viral_selection'].max()
print("Le score viable le plus bas est :", min_valide)
print("Le score viable le plus haut est :", max_valide)

extraction_seq = data['sequence'].tolist()
liste_seq = []
for i in extraction_seq:
    boucle = i.upper().replace('*', '')
    liste_seq.append(VP1_WT_1 + boucle + VP1_WT_2)


extraction_score = data['viral_selection'].replace(-np.inf, -15).tolist()
liste_score = []
for i in extraction_score:
    liste_score.append(i)

import random as rnd

#n = rnd.randint(0,len(liste_score))
print (liste_seq[1270])
print (liste_score[1270])



Le score viable le plus bas est : -11.176109295334715
Le score viable le plus haut est : 9.53645667061
MAADGYLPDWLEDTLSEGIRQWWKLKPGPPPPKPAERHKDDSRGLVLPGYKYLGPFNGLDKGEPVNEADAAALEHDKAYDRQLDSGDNPYLKYNHADAEFQERLKEDTSFGGNLGRAVFQAKKRVLEPLGLVEEPVKTAPGKKRPVEHSPVEPDSSSGTGKAGQQPARKRLNFGQTGDADSVPDPQPLGQPPAAPSGLGTNTMATGSGAPMADNNEGADGVGNSSGNWHCDSTWMGDRVITTSTRTWALPTYNNHLYKQISSQSGASNDNHYFGYSTPWGYFDFNRFHCHFSPRDWQRLINNNWGFRPKRLNFKLFNIQVKEVTQNDGTTTIANNLTSTVQVFTDSEYQLPYVLGSAHQGCLPPFPADVFMVPQYGYLTLNNGSQAVGRSSFYCLEYFPSQMLRTGNNFTFSYTFEDVPFHSSYAHSQSLDRLMNPLIDQYLYYLSRTNTPSGTTTQSRLQFSQAGASDIRDQSRNWLPGPCYRQQRVSKTSADNNNSEYSWTGATKYHLNGRDSLVNPGPAMASHKDDEEKFFPQSGVLIFGKQGSEKTNVDIEKVMITCEDRLKQTNPCMCEHHAVNVSEVCGPQTVHWVAMHQAATADVNTQGVLPGMVWQDRDVYLQGPIWAKIPHTDGHFHPSPLMGGFGLKHPPPQILIKNTPVPANPSTTFSAAKFASFITQYSTGQVSVEIEWELQKENSKRWNPEIQYTSNYNKSVNVDFTVDTNGVYSEPRPIGTRYLTRNL
-999999999.0


## 2. Traduction des séquences en vecteur

On utilise le modèle ESM-2 afin de rentrer une séquence, et d'en sortir un vecteur de 1280 nombres

In [ ]:

import esm
import torch

# 1. Sélection du hardware (ajout de 'mps' pour la puce Apple Silicon)
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

# 2. récupération directe de la variable 'alphabet'
model, alphabet = esm.pretrained.esm2_t6_8M_UR50D()

# 3. Initialisation du batch converter pour la tokenisation
batch_converter = alphabet.get_batch_converter()

# 4. Passage en mode évaluation et transfert sur le processeur (GPU/CPU)
model.eval()
model.to(device)

# Vérification : affichage du mapping des acides aminés
print(f"Modèle chargé sur : {device}")
print(alphabet.to_dict())

In [ ]:

#On va créer des tuples pour que notre Batch converter les utilise (il exige absolument un tuple avec un identifiant devant la séquence)

data_tuple = [(f"seq_{i}", seq) for i, seq in enumerate(liste_seq)]
print (len(data_tuple))


# maintenant, on utilise la fonction batch_converter pour convertir la liste de tuples en un vecteur !
batch_labels, batch_strs, batch_tokens = batch_converter(data_tuple)

print("Type de batch_tokens :", type(batch_tokens))
print("Dimensions du tenseur :", batch_tokens.shape)
print("Contenu brut du premier élément :\n", batch_tokens[0, :15])  # Affiche les 15 premiers acides aminés convertis

## 3. Modèle SciKit Learn ou PyTorch

Le modèle s'entraîne sur ce jeu de données vecteur/booléen. Il regarde le vecteur et prédit si le booléen est 0 ou 1

In [26]:


VP1_WT_1 = "MAADGYLPDWLEDTLSEGIRQWWKLKPGPPPPKPAERHKDDSRGLVLPGYKYLGPFNGLDKGEPVNEADAAALEHDKAYDRQLDSGDNPYLKYNHADAEFQERLKEDTSFGGNLGRAVFQAKKRVLEPLGLVEEPVKTAPGKKRPVEHSPVEPDSSSGTGKAGQQPARKRLNFGQTGDADSVPDPQPLGQPPAAPSGLGTNTMATGSGAPMADNNEGADGVGNSSGNWHCDSTWMGDRVITTSTRTWALPTYNNHLYKQISSQSGASNDNHYFGYSTPWGYFDFNRFHCHFSPRDWQRLINNNWGFRPKRLNFKLFNIQVKEVTQNDGTTTIANNLTSTVQVFTDSEYQLPYVLGSAHQGCLPPFPADVFMVPQYGYLTLNNGSQAVGRSSFYCLEYFPSQMLRTGNNFTFSYTFEDVPFHSSYAHSQSLDRLMNPLIDQYLYYLSRTNTPSGTTTQSRLQFSQAGASDIRDQSRNWLPGPCYRQQRVSKTSADNNNSEYSWTGATKYHLNGRDSLVNPGPAMASHKDDEEKFFPQSGVLIFGKQGSEKTNVDIEKVMIT"

WT_boucle28 = "DEEEIRTTNPVATEQYGSVSTNLQRGNR"

VP1_WT_2 = "QAATADVNTQGVLPGMVWQDRDVYLQGPIWAKIPHTDGHFHPSPLMGGFGLKHPPPQILIKNTPVPANPSTTFSAAKFASFITQYSTGQVSVEIEWELQKENSKRWNPEIQYTSNYNKSVNVDFTVDTNGVYSEPRPIGTRYLTRNL"

VP1_WT = VP1_WT_1 + WT_boucle28 + VP1_WT_2

import pandas as pd
import numpy as np
data = pd.read_csv('Variants.csv', sep=';')

# Afficher la liste exacte des noms de colonnes et les premières lignes
print("Colonnes détectées :", data.columns.tolist())
print(data.head(2))

extraction_seq = data['sequence'].tolist()
liste_seq = []
for i in extraction_seq:
    boucle = i.upper().replace('*', '')
    liste_seq.append(VP1_WT_1 + boucle + VP1_WT_2)


extraction_score = data['viral_selection'].replace(-np.inf, -15).tolist()
liste_score = []
for i in extraction_score:
    liste_score.append(i)

import random as rnd

#n = rnd.randint(0,len(liste_score))
print (liste_seq[1270])
print (liste_score[1270])




import esm
import torch

# 1. Sélection du hardware (ajout de 'mps' pour la puce Apple Silicon)
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

# 2. récupération directe de la variable 'alphabet'
model, alphabet = esm.pretrained.esm2_t6_8M_UR50D()

# 3. Initialisation du batch converter pour la tokenisation
batch_converter = alphabet.get_batch_converter()

# 4. Passage en mode évaluation et transfert sur le processeur (GPU/CPU)
model.eval()
model.to(device)

# Vérification : affichage du mapping des acides aminés
print(f"Modèle chargé sur : {device}")
print(alphabet.to_dict())




#On va créer des tuples pour que notre Batch converter les utilise (il exige absolument un tuple avec un identifiant devant la séquence)

data_tuple = [(f"seq_{i}", seq) for i, seq in enumerate(liste_seq)]
print (len(data_tuple))


# maintenant, on utilise la fonction batch_converter pour convertir la liste de tuples en un vecteur !
batch_labels, batch_strs, batch_tokens = batch_converter(data_tuple)

print("Type de batch_tokens :", type(batch_tokens))
print("Dimensions du tenseur :", batch_tokens.shape)
print("Contenu brut du premier élément :\n", batch_tokens[0, :15])  # Affiche les 15 premiers acides aminés convertis

from sklearn.svm import SVR
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import numpy as np
import torch

batch_size = 16
all_embeddings = []

print(f"Début de l'extraction sur : {device}")

for i in range(0, len(data_tuple), batch_size):
    batch = data_tuple[i : i + batch_size]

    # Tokenisation du petit lot
    _, _, batch_tokens = batch_converter(batch)
    batch_tokens = batch_tokens.to(device)

    # Passage dans ESM-2
    with torch.no_grad():
        results = model(batch_tokens, repr_layers=[6], return_contacts=False)

    embeddings_bruts = results["representations"][6]

    # Mean Pooling
    for j, (_, seq) in enumerate(batch):
        longueur_reelle = len(seq)
        tranche_proteine = embeddings_bruts[j, 1 : longueur_reelle + 1]
        vecteur_moyen = tranche_proteine.mean(dim=0)
        all_embeddings.append(vecteur_moyen.cpu().numpy())

    # Suivi de progression tous les 5 lots (80 séquences)
    if (i // batch_size) % 5 == 0:
        print(f"Progression : {min(i + batch_size, len(data_tuple))}/{len(data_tuple)} séquences traitées...")

X = np.array(all_embeddings)
y = np.array(liste_score)
print(f"Extraction terminée ! Matrice X : {X.shape} | Vecteur y : {y.shape}")

regr = make_pipeline(StandardScaler(), SVR(C=1.0, epsilon=0.2))
regr.fit(X, y)
print("Modèle SVR entraîné avec succès !")

Colonnes détectées : ['sequence', 'partition', 'mutation_sequence', 'num_mutations', 'num_edits', 'viral_selection', 'is_viable']
                       sequence partition             mutation_sequence  \
0  ADEEIRATNPIATEMYGSVSTNLQLGNR  designed  AD____A___I___M_________L___   
1  ADEEIRATNPVATEQYGSVSTNQQRQNR  designed  AD____A_______________Q__Q__   

   num_mutations  num_edits  viral_selection  is_viable  
0              6          6        -2.027259      False  
1              5          5        -0.429554       True  
MAADGYLPDWLEDTLSEGIRQWWKLKPGPPPPKPAERHKDDSRGLVLPGYKYLGPFNGLDKGEPVNEADAAALEHDKAYDRQLDSGDNPYLKYNHADAEFQERLKEDTSFGGNLGRAVFQAKKRVLEPLGLVEEPVKTAPGKKRPVEHSPVEPDSSSGTGKAGQQPARKRLNFGQTGDADSVPDPQPLGQPPAAPSGLGTNTMATGSGAPMADNNEGADGVGNSSGNWHCDSTWMGDRVITTSTRTWALPTYNNHLYKQISSQSGASNDNHYFGYSTPWGYFDFNRFHCHFSPRDWQRLINNNWGFRPKRLNFKLFNIQVKEVTQNDGTTTIANNLTSTVQVFTDSEYQLPYVLGSAHQGCLPPFPADVFMVPQYGYLTLNNGSQAVGRSSFYCLEYFPSQMLRTGNNFTFSYTFEDVPFHSSYAHSQSLDRLMNPLIDQYLYYLSRTNTPSGTTTQSRLQFSQAGASDI

In [ ]:

VP1_WT_1 = "MAADGYLPDWLEDTLSEGIRQWWKLKPGPPPPKPAERHKDDSRGLVLPGYKYLGPFNGLDKGEPVNEADAAALEHDKAYDRQLDSGDNPYLKYNHADAEFQERLKEDTSFGGNLGRAVFQAKKRVLEPLGLVEEPVKTAPGKKRPVEHSPVEPDSSSGTGKAGQQPARKRLNFGQTGDADSVPDPQPLGQPPAAPSGLGTNTMATGSGAPMADNNEGADGVGNSSGNWHCDSTWMGDRVITTSTRTWALPTYNNHLYKQISSQSGASNDNHYFGYSTPWGYFDFNRFHCHFSPRDWQRLINNNWGFRPKRLNFKLFNIQVKEVTQNDGTTTIANNLTSTVQVFTDSEYQLPYVLGSAHQGCLPPFPADVFMVPQYGYLTLNNGSQAVGRSSFYCLEYFPSQMLRTGNNFTFSYTFEDVPFHSSYAHSQSLDRLMNPLIDQYLYYLSRTNTPSGTTTQSRLQFSQAGASDIRDQSRNWLPGPCYRQQRVSKTSADNNNSEYSWTGATKYHLNGRDSLVNPGPAMASHKDDEEKFFPQSGVLIFGKQGSEKTNVDIEKVMIT"

WT_boucle28 = "DEEEIRTTNPVATEQYGSVSTNLQRGNR"

VP1_WT_2 = "QAATADVNTQGVLPGMVWQDRDVYLQGPIWAKIPHTDGHFHPSPLMGGFGLKHPPPQILIKNTPVPANPSTTFSAAKFASFITQYSTGQVSVEIEWELQKENSKRWNPEIQYTSNYNKSVNVDFTVDTNGVYSEPRPIGTRYLTRNL"

VP1_WT = VP1_WT_1 + WT_boucle28 + VP1_WT_2

import pandas as pd
import numpy as np
data = pd.read_csv('seq_test.csv')

seq_test = data['sequence'].tolist()
liste_seq_test = []
for i in seq_test:
    boucle = i.upper().replace('*', '')
    liste_seq_test.append(VP1_WT_1 + boucle + VP1_WT_2)


import esm
import torch

# 1. Sélection du hardware (ajout de 'mps' pour la puce Apple Silicon)
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

# 2. récupération directe de la variable 'alphabet'
model, alphabet = esm.pretrained.esm2_t6_8M_UR50D()

# 3. Initialisation du batch converter pour la tokenisation
batch_converter = alphabet.get_batch_converter()

# 4. Passage en mode évaluation et transfert sur le processeur (GPU/CPU)
model.eval()
model.to(device)

# Vérification : affichage du mapping des acides aminés
print(f"Modèle chargé sur : {device}")
print(alphabet.to_dict())




#On va créer des tuples pour que notre Batch converter les utilise (il exige absolument un tuple avec un identifiant devant la séquence)

data_tuple = [(f"seq_{i}", seq) for i, seq in enumerate(liste_seq_test)]
print (len(data_tuple))


# maintenant, on utilise la fonction batch_converter pour convertir la liste de tuples en un vecteur !
batch_labels, batch_strs, batch_tokens = batch_converter(data_tuple)

print("Type de batch_tokens :", type(batch_tokens))
print("Dimensions du tenseur :", batch_tokens.shape)
print("Contenu brut du premier élément :\n", batch_tokens[0, :15])  # Affiche les 15 premiers acides aminés convertis


import numpy as np
import pandas as pd
import torch

# 1. EXTRACTION DES EMBEDDINGS POUR LE JEU DE TEST
batch_size = 16
all_embeddings_test = []

print("Début de l'extraction sur le jeu de test...")

for i in range(0, len(data_tuple), batch_size):
    batch = data_tuple[i : i + batch_size]

    # Tokenisation du petit lot
    _, _, batch_tokens = batch_converter(batch)
    batch_tokens = batch_tokens.to(device)

    # Passage dans les couches d'attention d'ESM-2
    with torch.no_grad():
        results = model(batch_tokens, repr_layers=[6], return_contacts=False)

    embeddings_bruts = results["representations"][6]

    # Mean Pooling
    for j, (_, seq) in enumerate(batch):
        longueur_reelle = len(seq)
        tranche_proteine = embeddings_bruts[j, 1 : longueur_reelle + 1]
        vecteur_moyen = tranche_proteine.mean(dim=0)
        all_embeddings_test.append(vecteur_moyen.cpu().numpy())

# 2. CONSTITUTION DE LA MATRICE DE TEST
X_test = np.array(all_embeddings_test)
print(f"Matrice X_test prête pour l'inférence : {X_test.shape}")  # Doit afficher (20, 320)

# 3. PRÉDICTION DES SCORES DE SÉLECTION VIRALE
scores_predits = regr.predict(X_test)

# 4. CONSTITUTION DU TABLEAU DE RÉSULTATS
resultats_df = pd.DataFrame({
    'mutation_boucle28': seq_test,
    'score_viral_predit': scores_predits
})

# Tri du variant le plus performant au moins viable
resultats_df = resultats_df.sort_values(by='score_viral_predit', ascending=False).reset_index(drop=True)

print("\n--- CLASSEMENT DES VARIANTS PRÉDITS ---")
print(resultats_df)


## A. Génération des séquences mutées

On prend une mutation qui est différente au maximum de la séquence de référence, puis on la fais passer dans ESM-2 pour obtenir un vecteur de 1280 nombres

Pour cela, on génère un nombre aléatoire entre 1 et 27, avec 50% de chances que ce soit 1, 25% que ce soit 2, 12.5% que ce soit 3 (par exemple, et à réfléchir car il faut qu'il y ait quand même pas mal de mutations donc on va peut-être partir en gaussienne, ou la médiane serait 4 mutations, ou 3,...)

On aura alors une grande quantité de séquences plus ou moins mutées, et à chaque fois, il devra y avoir un score de mutation. A réfléchir : Soit il est sur 27 (les 27AA), et le score augmente de 1 par mutation ? Ou alors, le score est calculé par des maths ou un autre modèle qui prend en compte la conformation ? Ou à partir du vecteur ESM-2 ?

Donc on a pleins de séquences mutées qui ont un score de différence. Le but sera d'avoir un score de différence élevé, ET un score d'assemblage élevé (ou d'être positif à l'assemblage)

## B. Test de la séquence d'intérêt à l'assemblage

On fais passer la séquence à tester dans le modèle afin de prédire son bon assemblage

On répète ces deux étapes A et B à l'infini, et on stock dans une liste de la plus différente à la moins les 15 séquences les plus différentes qui sont 1 pour "assemblage"